# MAZEWARD Colab 制御サーバー（ngrok）

このノートブックは Colab 上で学習を起動し、ローカルGUIから開始・停止・状況確認できるようにするための最小セットです。

手順: ① Setup → ② Start Control Server → ③ Open ngrok

最後のセルで、GUI に貼る URL と API トークンが出ます。

In [ ]:
import os, shutil, secrets
from google.colab import drive

DRIVE_FOLDER = "mazeward_colab_rl_ai"
COLAB_DIR = "/content/mazeward_colab_rl_ai"

drive.mount("/content/drive", force_remount=True)
src = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
assert os.path.exists(src), f"Drive に {src} がありません。先に GUI の『📤 コードを Colab へ送信』を実行してください"

if os.path.exists(COLAB_DIR):
    shutil.rmtree(COLAB_DIR)
shutil.copytree(src, COLAB_DIR, ignore=shutil.ignore_patterns("models", "replays", "__pycache__", "*.ipynb", "API.txt"))
print("コードを更新しました:", COLAB_DIR)

api_path = os.path.join(COLAB_DIR, "ai", "API.txt")
os.makedirs(os.path.dirname(api_path), exist_ok=True)
if not os.path.exists(api_path):
    token = secrets.token_hex(16)
    with open(api_path, "w", encoding="utf-8") as f:
        f.write(token + "\n")
else:
    token = open(api_path, encoding="utf-8").read().strip()

print("API トークン:", token)
print("Drive 側保存先:", api_path)

In [ ]:
import subprocess, os, sys, time
COLAB_DIR = "/content/mazeward_colab_rl_ai"
log = open("/content/mazeward_control.log", "w", encoding="utf-8")
proc = subprocess.Popen(
    [sys.executable, f"{COLAB_DIR}/ai/mazeward_colab_control.py", "--port", "5558"],
    cwd=COLAB_DIR, stdout=log, stderr=subprocess.STDOUT,
)
time.sleep(3)
print("制御サーバー起動中…", proc.pid)
print("ログ:", "/content/mazeward_control.log")

In [ ]:
import os, subprocess, getpass, time, requests
try:
    from pyngrok import ngrok
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import ngrok

auth = os.environ.get("NGROK_AUTHTOKEN") or ""
if not auth:
    auth = getpass.getpass("ngrok authtoken を入力: ")
if auth:
    ngrok.set_auth_token(auth)

tunnel = ngrok.connect(5558)
public = tunnel.public_url if tunnel else None
token = open("/content/mazeward_colab_rl_ai/ai/API.txt", encoding="utf-8").read().strip()
print("=" * 64)
print("GUI に貼る値")
print("ngrok URL :", public or "(取得失敗)")
print("API Token :", token)
print("=" * 64)

if public:
    r = requests.get(public + "/healthz", headers={"X-API-Token": token}, timeout=10)
    print("healthz:", r.status_code, r.text[:200])
else:
    print("URL が取得できていません。ngrok authtoken を確認してください。")